# Notebook 01 - AI Search Por Usuario

Crea AI Search en el RG del participante y carga un indice vectorial de conocimiento.

In [18]:
import json
import subprocess
import requests
from pathlib import Path
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

with open('workshop_config.json', 'r') as f:
    cfg = json.load(f)

assert cfg.get('setup_verified'), 'Ejecuta primero notebook 00'

SUBSCRIPTION_ID = cfg['azure_subscription_id']
LOCATION = cfg['azure_location']
USER_RG_NAME = cfg['user_rg_name']
RESOURCE_PREFIX = cfg['resource_prefix']
SEARCH_SERVICE_NAME = f"{cfg['user_alias']}srch".replace('-', '')[:48]
SEARCH_INDEX_NAME = f"{RESOURCE_PREFIX}-kb".lower()[:128]

def run(cmd):
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(p.stderr or p.stdout)
    return p.stdout.strip()

run(['az', 'account', 'set', '--subscription', SUBSCRIPTION_ID])
print('Search service:', SEARCH_SERVICE_NAME)
print('Index:', SEARCH_INDEX_NAME)

Search service: user01srch
Index: contoso-user01-kb


In [19]:
# Asegura que exista el RG por usuario para este laboratorio
run([
    'az', 'group', 'create',
    '--name', USER_RG_NAME,
    '--location', LOCATION,
    '-o', 'none'
])

probe = subprocess.run([
    'az', 'search', 'service', 'show',
    '--name', SEARCH_SERVICE_NAME,
    '--resource-group', USER_RG_NAME
], capture_output=True, text=True)

if probe.returncode != 0:
    run([
        'az', 'search', 'service', 'create',
        '--name', SEARCH_SERVICE_NAME,
        '--resource-group', USER_RG_NAME,
        '--location', LOCATION,
        '--sku', 'basic'
    ])
    print('Search creado')
else:
    print('Search ya existia')

search_admin_key = run([
    'az', 'search', 'admin-key', 'show',
    '--service-name', SEARCH_SERVICE_NAME,
    '--resource-group', USER_RG_NAME,
    '--query', 'primaryKey', '-o', 'tsv'
].copy())
search_endpoint = f"https://{SEARCH_SERVICE_NAME}.search.windows.net"
search_api_version = '2024-07-01'

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(credential, 'https://cognitiveservices.azure.com/.default')
aoai = AzureOpenAI(
    azure_endpoint=cfg['azure_openai_endpoint'],
    azure_ad_token_provider=token_provider,
    api_version=cfg['api_version']
)

embedding_deployment = cfg.get('embedding_model_deployment', '').strip()
USE_VECTOR_SEARCH = True
VECTOR_DIMENSIONS = None

def _find_embedding_deployments_from_endpoint(endpoint: str):
    normalized = endpoint.rstrip('/')
    account_tsv = run([
        'az', 'cognitiveservices', 'account', 'list',
        '--query', f"[?properties.endpoint=='{normalized}'].[name,resourceGroup] | [0]",
        '-o', 'tsv'
    ])
    if not account_tsv:
        return []
    account_name, account_rg = account_tsv.split('\t')
    deployments_tsv = run([
        'az', 'cognitiveservices', 'account', 'deployment', 'list',
        '--name', account_name,
        '--resource-group', account_rg,
        '--query', "[?contains(properties.model.name, 'embedding')].name",
        '-o', 'tsv'
    ])
    return [d.strip() for d in deployments_tsv.splitlines() if d.strip()]

if embedding_deployment:
    try:
        probe_emb = aoai.embeddings.create(model=embedding_deployment, input='ping')
        VECTOR_DIMENSIONS = len(probe_emb.data[0].embedding)
    except Exception as ex:
        if 'DeploymentNotFound' not in str(ex):
            raise
        candidates = _find_embedding_deployments_from_endpoint(cfg['azure_openai_endpoint'])
        if candidates:
            embedding_deployment = candidates[0]
            cfg['embedding_model_deployment'] = embedding_deployment
            probe_emb = aoai.embeddings.create(model=embedding_deployment, input='ping')
            VECTOR_DIMENSIONS = len(probe_emb.data[0].embedding)
            print(f"Embedding deployment configurado no existe. Usando fallback: {embedding_deployment}")
        else:
            USE_VECTOR_SEARCH = False
            embedding_deployment = ''
            print('No hay deployment de embeddings disponible. Se usara indice textual.')
else:
    USE_VECTOR_SEARCH = False
    print('No se definio embedding_model_deployment. Se usara indice textual.')

print('Endpoint:', search_endpoint)
print('Vector search habilitado:', USE_VECTOR_SEARCH)
if USE_VECTOR_SEARCH:
    print('Embedding deployment:', embedding_deployment)
    print('Vector dimensions:', VECTOR_DIMENSIONS)

Search ya existia
Endpoint: https://user01srch.search.windows.net
Vector search habilitado: True
Embedding deployment: text-embedding-3-small
Vector dimensions: 1536


In [23]:
fields = [
  {'name': 'id', 'type': 'Edm.String', 'key': True, 'searchable': False, 'filterable': True, 'retrievable': True},
  {'name': 'source', 'type': 'Edm.String', 'searchable': True, 'filterable': True, 'retrievable': True},
  {'name': 'content', 'type': 'Edm.String', 'searchable': True, 'retrievable': True}
 ]

index_schema = {'name': SEARCH_INDEX_NAME, 'fields': fields}

if USE_VECTOR_SEARCH:
  index_schema['fields'].append({
      'name': 'contentVector',
      'type': 'Collection(Edm.Single)',
      'searchable': True,
      'retrievable': False,
      'dimensions': VECTOR_DIMENSIONS,
      'vectorSearchProfile': 'vprofile'
  })
  index_schema['vectorSearch'] = {
      'algorithms': [{'name': 'halgo', 'kind': 'hnsw'}],
      'profiles': [{'name': 'vprofile', 'algorithm': 'halgo'}]
  }

resp = requests.put(
    f"{search_endpoint}/indexes/{SEARCH_INDEX_NAME}?api-version={search_api_version}",
    headers={'Content-Type': 'application/json', 'api-key': search_admin_key},
    json=index_schema
)
if resp.status_code not in (200, 201, 204):
    raise RuntimeError(resp.text or f'Error creando indice. HTTP {resp.status_code}')
print('Indice listo. Modo vector:', USE_VECTOR_SEARCH)

Indice listo. Modo vector: True


In [21]:
def chunk_text(text, size=1200, overlap=150):
    out = []
    i = 0
    while i < len(text):
        out.append(text[i:i+size])
        i += max(1, size-overlap)
    return out

docs = [
    ('faq', Path('data/faq_contoso.md').read_text(encoding='utf-8')),
    ('politicas', Path('data/politicas_credito.md').read_text(encoding='utf-8'))
]

items = []
for source, text in docs:
    for idx, part in enumerate(chunk_text(text)):
        item = {
            'id': f'{source}-{idx}',
            'source': source,
            'content': part
        }
        if USE_VECTOR_SEARCH:
            emb = aoai.embeddings.create(model=embedding_deployment, input=part)
            item['contentVector'] = emb.data[0].embedding
        items.append(item)

payload = {'value': [{'@search.action': 'upload', **d} for d in items]}
ru = requests.post(
    f"{search_endpoint}/indexes/{SEARCH_INDEX_NAME}/docs/index?api-version={search_api_version}",
    headers={'Content-Type': 'application/json', 'api-key': search_admin_key},
    json=payload
)
if ru.status_code not in (200, 201):
    raise RuntimeError(ru.text)
print('Chunks cargados:', len(items))

Chunks cargados: 10


In [24]:
if 'search_endpoint' not in globals() or 'search_admin_key' not in globals():
    raise RuntimeError('AI Search no inicializado. Ejecuta celdas previas y corrige errores.')
if 'items' not in globals() or not items:
    raise RuntimeError('No hay documentos indexados. Ejecuta la carga de chunks antes de guardar estado.')

cfg['ai_search'] = {
    'service_name': SEARCH_SERVICE_NAME,
    'index_name': SEARCH_INDEX_NAME,
    'endpoint': search_endpoint,
    'admin_key': search_admin_key,
    'api_version': search_api_version,
    'use_vector_search': USE_VECTOR_SEARCH
}
if USE_VECTOR_SEARCH:
    cfg['ai_search']['vector_dimensions'] = VECTOR_DIMENSIONS
    cfg['ai_search']['embedding_model_deployment'] = embedding_deployment

cfg['ai_search_ready'] = True
with open('workshop_config.json', 'w') as f:
    json.dump(cfg, f, indent=2)
print('OK AI Search configurado')

OK AI Search configurado
